In [2]:
!pip install langchain langchain-openai langchain-community pydantic pypdf


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()  # loads OPENROUTER_API_KEY from .env if present

if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")



# M1.Ex1: Models
### Task 1: Text Generation Temperature

In [4]:
import os
from langchain_openai import ChatOpenAI

MODEL = "nvidia/nemotron-3-nano-30b-a3b:free"

def make_llm(temperature: float = 0.7):
    return ChatOpenAI(
        model=MODEL,
        temperature=temperature,
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url="https://openrouter.ai/api/v1",
    )

In [5]:
cold = make_llm(temperature=0.0)
hot  = make_llm(temperature=1.5)

prompt = "Why is the sky blue? Be poetic in your response."

print("--- COLD (T=0.0) ---")
print(cold.invoke(prompt).content)

print("\n--- HOT (T=1.5) ---")
print(hot.invoke(prompt).content)

--- COLD (T=0.0) ---
The sky is a vast, ever‑changing canvas, and its hue is the world’s most delicate brushstroke.  
When the sun climbs high, its light is a chorus of colors — golden amber, fierce violet, soft rose — each note traveling the same invisible highway of air.  

But the atmosphere is a shy, restless audience. It loves the high‑pitched notes and shuns the low.  Short‑wavelength blues and violets, quick‑tempered and eager, bounce off the tiny molecules of nitrogen and oxygen with a playful flicker. They scatter in every direction, spilling their cerulean secret across the dome above us.  

The longer, steadier reds and yellows, content to wander straight on, slip through the gaps untouched, leaving the sky awash in the cool, crystalline whisper of blue.  

So the heavens wear their sapphire veil not because they are painted that way, but because the very breath of the world chooses to dance with the shortest, most fleeting light — turning the endless expanse into a poem wri

In [6]:
print(cold.invoke("Why is the sky blue? Format your answer as JSON with keys 'reason' and 'short_explanation'.").content)

{
  "reason": "Sunlight contains all colors, but when it passes through Earth's atmosphere, shorter blue wavelengths are scattered in all directions more efficiently than longer red wavelengths. This preferential scattering makes the sky appear blue to our eyes.",
  "short_explanation": "Blue light scatters more in the atmosphere, giving the sky its blue color."
}


### Task 2 : Task 2: Sentiment Analysis
Define the shape you want. The model is then constrained (via the OpenRouter / function-calling layer) to return exactly that shape:

In [7]:
from pydantic import BaseModel, Field
from typing import Literal

class Sentiment(BaseModel):
    label: Literal["positive", "neutral", "negative"] = Field(
        description="Overall sentiment of the input sentence."
    )

llm = make_llm(temperature=0)
sentiment_llm = llm.with_structured_output(Sentiment)

sentences = [
    "Kindness creates lasting joy.",
    "Success rewards persistent effort.",
    "I love Sunlight. It warms the skin.",
    "Pessemestic all the time.",
    "The storm caused damage!",
    "The clock ticks steadily.",
]

for s in sentences:
    result = sentiment_llm.invoke(f"Classify the sentiment of this sentence: {s}")
    print(f"{result.label:>10}  ←  {s}")

  positive  ←  Kindness creates lasting joy.
  positive  ←  Success rewards persistent effort.
  positive  ←  I love Sunlight. It warms the skin.
  negative  ←  Pessemestic all the time.
  negative  ←  The storm caused damage!
   neutral  ←  The clock ticks steadily.


### Task 3 — Multi-label categorization

In [8]:
import json
from typing import List

class Tags(BaseModel):
    tags: List[Literal["cars", "shopping", "sports", "study", "work"]] = Field(
        description="All tags that apply to the message. Can be empty."
    )

_llm = make_llm(temperature=0)
tagger = _llm.with_structured_output(Tags)
VALID_TAGS = {"cars", "shopping", "sports", "study", "work"}

messages = [
    "That restoration looks incredible; you have a real talent for mechanics.",
    "I found the perfect gift today! The staff was incredibly helpful.",
    "Great game today! Your teamwork was the key to that victory.",
    "Learning together helped me finally grasp these concepts. Thank you!",
]

for m in messages:
    try:
        result = tagger.invoke(m)
    except Exception:
        # Free model sometimes embeds control chars in JSON output; parse manually.
        raw = _llm.invoke(
            "Return ONLY a JSON object with key \"tags\". "
            "Pick applicable tags from: cars, shopping, sports, study, work. "
            f"Message: {m}"
        ).content
        # Remove control characters using ord() — no regex or escape sequences needed
        clean = ''.join(c for c in raw if ord(c) >= 32)
        start, end = clean.find('{'), clean.rfind('}')
        tags_list = []
        if start != -1 and end != -1:
            try:
                data = json.loads(clean[start:end + 1])
                tags_list = [t for t in data.get('tags', []) if t in VALID_TAGS]
            except Exception:
                pass
        result = Tags(tags=tags_list)
    print(result.tags, '<-', m)

['cars'] <- That restoration looks incredible; you have a real talent for mechanics.
['shopping', 'sports', 'sports'] <- I found the perfect gift today! The staff was incredibly helpful.
['sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports', 'sports'] <- Great game today! Your teamwork was the key to that victory.
['cars', 'cars'] <- Learning together helped me finally grasp these concepts. Thank you!


### Task 4 — Resume data extraction

In [10]:
from typing import List, Optional
from langchain_community.document_loaders import PyPDFLoader

class Experience(BaseModel):
    company: str
    role: str
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    summary: Optional[str] = None

class Resume(BaseModel):
    candidate_name: str = Field(description="Full name of the candidate")
    email: Optional[str] = None
    phone: Optional[str] = None
    skills: List[str] = Field(default_factory=list)
    experience: List[Experience] = Field(default_factory=list)
    education: List[str] = Field(default_factory=list)

# 1. Load PDF
pages = PyPDFLoader("resume.pdf").load()
resume_text = "\n\n".join(p.page_content for p in pages)

# 2. Extract
extractor = make_llm(temperature=0).with_structured_output(Resume)
parsed = extractor.invoke(
    f"Extract structured information from this resume:\n\n{resume_text}"
)

print(parsed.model_dump_json(indent=2))

{
  "candidate_name": "Moath Al-Harthi",
  "email": "moathalharthi11@gmail.com",
  "phone": "0533700845",
  "skills": [
    "IT Operations Management",
    "System Administration (AD)",
    "Communication & Stakeholder Support",
    "Technical Troubleshooting",
    "Requirements Gathering",
    "Process Improvement",
    "Incident & Problem Management",
    "Service Desk & Ticketing Tools",
    "Business Analysis"
  ],
  "experience": [
    {
      "company": "National Gas & Industrialization Co. (GASCO)",
      "role": "IT Specialist",
      "start_date": "07/2024",
      "end_date": "Present",
      "summary": "Led daily IT operations including system monitoring, user access management, and incident resolution to ensure high service availability. Analyzed recurring technical issues, documented root causes, and proposed long-term solutions that helped reduce repeated incidents. Collaborated with business units to gather requirements, understand workflow challenges, and translate them 

### Task 5 — Tools (the punchline)

In [11]:
from typing import Dict, Any

# Step 1: actual Python functions
def add(a, b): return a + b
def subtract(a, b): return a - b
def multiply(a, b): return a * b
def divide(a, b): return a / b

TOOLS = {"add": add, "subtract": subtract, "multiply": multiply, "divide": divide}

# Step 2: describe them in the system prompt
system_prompt = """
You are a math assistant. You have access to these tools:

def add(a, b):       # returns a + b
def subtract(a, b):  # returns a - b
def multiply(a, b):  # returns a * b
def divide(a, b):    # returns a / b

Return the tool name and the arguments needed to solve the user's request.
"""

# Step 3: schema for the model's reply
class ToolCall(BaseModel):
    tool_name: Literal["add", "subtract", "multiply", "divide"]
    arguments: Dict[str, Any] = Field(description="Keyword args for the tool, e.g. {'a': 2, 'b': 5}")

# Step 4: ask
router = make_llm(temperature=0).with_structured_output(ToolCall)

question = "what is two plus 5"
call = router.invoke(f"{system_prompt}\n\nUser: {question}")
print("Model decided:", call)

# Step 5: actually execute
result = TOOLS[call.tool_name](**call.arguments)
print(f"Answer: {result}")

Model decided: tool_name='add' arguments={'a': 2, 'b': 5}
Answer: 7
